In [1]:
# Lire depuis MinIO avec PySpark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark import StorageLevel

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("HospitalOptimizer-ETL-MinIO") \
    .config("spark.driver.memory", "4g") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://hospital-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "hospital") \
    .config("spark.hadoop.fs.s3a.secret.key", "hospital123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Test lecture depuis MinIO
test = spark.read.parquet("s3a://bronze/parquet/patients.parquet")
print(f"✅ MinIO OK — {test.count():,} lignes lues")

✅ MinIO OK — 50,000 lignes lues


In [8]:
BASE = "s3a://bronze/mimic_synthetic"

# Tables prioritaires pour nos 3 modèles
patients      = spark.read.csv(f"{BASE}/PATIENTS.csv",    header=True, inferSchema=True)
admissions    = spark.read.csv(f"{BASE}/ADMISSIONS.csv",  header=True, inferSchema=True)
icustays      = spark.read.csv(f"{BASE}/ICUSTAYS.csv",    header=True, inferSchema=True)
labevents     = spark.read.csv(f"{BASE}/LABEVENTS.csv",   header=True, inferSchema=True)
prescriptions = spark.read.csv(f"{BASE}/PRESCRIPTIONS.csv", header=True, inferSchema=True)
transfers     = spark.read.csv(f"{BASE}/TRANSFERS.csv",   header=True, inferSchema=True)
services      = spark.read.csv(f"{BASE}/SERVICES.csv",    header=True, inferSchema=True)

print("✅ Tables chargées depuis MinIO :")
print(f"   PATIENTS      : {patients.count():>10,} lignes")
print(f"   ADMISSIONS    : {admissions.count():>10,} lignes")
print(f"   ICUSTAYS      : {icustays.count():>10,} lignes")
print(f"   LABEVENTS     : {labevents.count():>10,} lignes")
print(f"   PRESCRIPTIONS : {prescriptions.count():>10,} lignes")
print(f"   TRANSFERS     : {transfers.count():>10,} lignes")
print(f"   SERVICES      : {services.count():>10,} lignes")

Py4JJavaError: An error occurred while calling o59.csv.
: java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2688)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3431)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.execution.datasources.DataSource$.$anonfun$checkAndGlobPathIfNecessary$1(DataSource.scala:724)
	at scala.collection.immutable.List.map(List.scala:293)
	at org.apache.spark.sql.execution.datasources.DataSource$.checkAndGlobPathIfNecessary(Dat

In [3]:
# Sauvegarder en Parquet dans MinIO
# Maintenant on convertit les CSV en Parquet dans le bucket bronze — c'est plus rapide à lire pour PySpark.

tables = {
    "patients":      patients,
    "admissions":    admissions,
    "icustays":      icustays,
    "labevents":     labevents,
    "prescriptions": prescriptions,
    "transfers":     transfers,
    "services":      services,
}

print("Conversion CSV → Parquet dans MinIO bronze...\n")

for name, df in tables.items():
    path = f"s3a://bronze/parquet/{name}.parquet"
    df.write.mode("overwrite").parquet(path)
    print(f"✅ {name:<15} → {path}")

print("\n✅ Bronze Parquet prêt dans MinIO")

Conversion CSV → Parquet dans MinIO bronze...

✅ patients        → s3a://bronze/parquet/patients.parquet
✅ admissions      → s3a://bronze/parquet/admissions.parquet
✅ icustays        → s3a://bronze/parquet/icustays.parquet
✅ labevents       → s3a://bronze/parquet/labevents.parquet
✅ prescriptions   → s3a://bronze/parquet/prescriptions.parquet
✅ transfers       → s3a://bronze/parquet/transfers.parquet
✅ services        → s3a://bronze/parquet/services.parquet

✅ Bronze Parquet prêt dans MinIO


In [9]:
from pyspark import StorageLevel
from pyspark.sql import functions as F

# Tables petites → cache RAM
patients_b      = spark.read.parquet("s3a://bronze/parquet/patients.parquet").cache()
admissions_b    = spark.read.parquet("s3a://bronze/parquet/admissions.parquet").cache()
icustays_b      = spark.read.parquet("s3a://bronze/parquet/icustays.parquet").cache()
prescriptions_b = spark.read.parquet("s3a://bronze/parquet/prescriptions.parquet").cache()
transfers_b     = spark.read.parquet("s3a://bronze/parquet/transfers.parquet").cache()
services_b      = spark.read.parquet("s3a://bronze/parquet/services.parquet").cache()

# LABEVENTS → grosse table → RAM + Disque
labevents_b = spark.read.parquet("s3a://bronze/parquet/labevents.parquet") \
    .persist(StorageLevel.MEMORY_AND_DISK)

# Forcer le chargement en cache
print("Chargement en cache...\n")
print(f"✅ patients      : {patients_b.count():>10,} lignes")
print(f"✅ admissions    : {admissions_b.count():>10,} lignes")
print(f"✅ icustays      : {icustays_b.count():>10,} lignes")
print(f"✅ prescriptions : {prescriptions_b.count():>10,} lignes")
print(f"✅ transfers     : {transfers_b.count():>10,} lignes")
print(f"✅ services      : {services_b.count():>10,} lignes")
print(f"✅ labevents     : {labevents_b.count():>10,} lignes")
print("\n✅ Toutes les tables en cache — prêtes pour l'exploration")

Py4JJavaError: An error occurred while calling o64.parquet.
: java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2688)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3431)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.execution.datasources.DataSource$.$anonfun$checkAndGlobPathIfNecessary$1(DataSource.scala:724)
	at scala.collection.immutable.List.map(List.scala:293)
	at org.apache.spark.sql.execution.datasources.DataSource$.checkAndGlobPathIfNecessary

In [6]:
# Exploration de chaque table
tables_bronze = {
    "patients":      patients_b,
    "admissions":    admissions_b,
    "icustays":      icustays_b,
    "labevents":     labevents_b,
    "prescriptions": prescriptions_b,
    "transfers":     transfers_b,
    "services":      services_b,
}

for name, df in tables_bronze.items():
    print("=" * 60)
    print(f"TABLE : {name.upper()}")
    print("=" * 60)
    print(f"Lignes   : {df.count():,}")
    print(f"Colonnes : {len(df.columns)}")
    print(f"Schema   : {df.columns}")

    # Valeurs manquantes
    missing = []
    for col in df.columns:
        n_null = df.filter(F.col(col).isNull()).count()
        if n_null > 0:
            pct = n_null / df.count() * 100
            missing.append((col, n_null, pct))

    if missing:
        print("\nValeurs manquantes :")
        for col, n, pct in missing:
            print(f"  {col:<30} : {n:>8,} ({pct:.1f}%)")
    else:
        print("\n✅ Aucune valeur manquante")
    print()

TABLE : PATIENTS
Lignes   : 50,000
Colonnes : 8
Schema   : ['ROW_ID', 'SUBJECT_ID', 'GENDER', 'DOB', 'DOD', 'DOD_HOSP', 'DOD_SSN', 'EXPIRE_FLAG']

Valeurs manquantes :
  DOD                            :   43,911 (87.8%)
  DOD_HOSP                       :   45,782 (91.6%)
  DOD_SSN                        :   46,973 (93.9%)

TABLE : ADMISSIONS
Lignes   : 80,000
Colonnes : 19
Schema   : ['ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'ADMITTIME', 'DISCHTIME', 'DEATHTIME', 'ADMISSION_TYPE', 'ADMISSION_LOCATION', 'DISCHARGE_LOCATION', 'INSURANCE', 'LANGUAGE', 'RELIGION', 'MARITAL_STATUS', 'ETHNICITY', 'EDREGTIME', 'EDOUTTIME', 'DIAGNOSIS', 'HOSPITAL_EXPIRE_FLAG', 'HAS_CHARTEVENTS_DATA']

Valeurs manquantes :
  DEATHTIME                      :   73,502 (91.9%)
  EDREGTIME                      :   47,780 (59.7%)
  EDOUTTIME                      :   47,780 (59.7%)

TABLE : ICUSTAYS
Lignes   : 100,000
Colonnes : 12
Schema   : ['ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'DBSOURCE', 'FIRST_CAREUNIT', '

In [7]:
print("Nettoyage Silver...\n")

# ── PATIENTS : aucun nettoyage ────────────────────────────────────
patients_s = patients_b
print(f"✅ patients      : {patients_s.count():>10,} lignes")

# ── ADMISSIONS : aucun nettoyage ──────────────────────────────────
admissions_s = admissions_b
print(f"✅ admissions    : {admissions_s.count():>10,} lignes")

# ── ICUSTAYS : aucun nettoyage ────────────────────────────────────
icustays_s = icustays_b
print(f"✅ icustays      : {icustays_s.count():>10,} lignes")

# ── LABEVENTS : VALUENUM null → 0, FLAG null → NORMAL ────────────
labevents_s = labevents_b \
    .fillna(0,        subset=["VALUENUM"]) \
    .fillna("NORMAL", subset=["FLAG"])
print(f"✅ labevents     : {labevents_s.count():>10,} lignes")

# ── PRESCRIPTIONS : aucun nettoyage ──────────────────────────────
prescriptions_s = prescriptions_b
print(f"✅ prescriptions : {prescriptions_s.count():>10,} lignes")

# ── TRANSFERS : aucun nettoyage ───────────────────────────────────
transfers_s = transfers_b
print(f"✅ transfers     : {transfers_s.count():>10,} lignes")

# ── SERVICES : aucun nettoyage ────────────────────────────────────
services_s = services_b
print(f"✅ services      : {services_s.count():>10,} lignes")

Nettoyage Silver...

✅ patients      :     50,000 lignes
✅ admissions    :     80,000 lignes
✅ icustays      :    100,000 lignes
✅ labevents     :  7,000,000 lignes
✅ prescriptions :  1,500,000 lignes
✅ transfers     :    150,000 lignes
✅ services      :    120,000 lignes


In [14]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark import StorageLevel

# Arrêter la session existante si elle existe
try:
    spark.stop()
except:
    pass

# Recréer Spark avec config MinIO
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("HospitalOptimizer-ETL-MinIO") \
    .config("spark.driver.memory", "4g") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://hospital-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "hospital") \
    .config("spark.hadoop.fs.s3a.secret.key", "hospital123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark {spark.version} connecté à MinIO : oK")

✅ Spark 3.5.0 connecté à MinIO : oK


In [2]:
# ── Recharger depuis Bronze ───────────────────────────────────────
patients_b      = spark.read.parquet("s3a://bronze/parquet/patients.parquet").cache()
admissions_b    = spark.read.parquet("s3a://bronze/parquet/admissions.parquet").cache()
icustays_b      = spark.read.parquet("s3a://bronze/parquet/icustays.parquet").cache()
labevents_b     = spark.read.parquet("s3a://bronze/parquet/labevents.parquet").persist(StorageLevel.MEMORY_AND_DISK)
prescriptions_b = spark.read.parquet("s3a://bronze/parquet/prescriptions.parquet").cache()
transfers_b     = spark.read.parquet("s3a://bronze/parquet/transfers.parquet").cache()
services_b      = spark.read.parquet("s3a://bronze/parquet/services.parquet").cache()

# ── Nettoyage Silver ──────────────────────────────────────────────
patients_s      = patients_b
admissions_s    = admissions_b
icustays_s      = icustays_b
labevents_s     = labevents_b \
    .fillna(0,        subset=["VALUENUM"]) \
    .fillna("NORMAL", subset=["FLAG"])
prescriptions_s = prescriptions_b
transfers_s     = transfers_b
services_s      = services_b

# ── Sauvegarder Silver dans MinIO ─────────────────────────────────
silver_tables = {
    "patients":      patients_s,
    "admissions":    admissions_s,
    "icustays":      icustays_s,
    "labevents":     labevents_s,
    "prescriptions": prescriptions_s,
    "transfers":     transfers_s,
    "services":      services_s,
}

print("Sauvegarde Silver dans MinIO...\n")
for name, df in silver_tables.items():
    path = f"s3a://silver/parquet/{name}.parquet"
    df.write.mode("overwrite").parquet(path)
    print(f"✅ {name:<15} → silver/parquet/{name}.parquet")

print("\n✅ Silver complet dans MinIO")

Sauvegarde Silver dans MinIO...

✅ patients        → silver/parquet/patients.parquet
✅ admissions      → silver/parquet/admissions.parquet
✅ icustays        → silver/parquet/icustays.parquet
✅ labevents       → silver/parquet/labevents.parquet
✅ prescriptions   → silver/parquet/prescriptions.parquet
✅ transfers       → silver/parquet/transfers.parquet
✅ services        → silver/parquet/services.parquet

✅ Silver complet dans MinIO


In [3]:
from pyspark.sql.window import Window

# ── Lire depuis Silver ────────────────────────────────────────────
patients_s      = spark.read.parquet("s3a://silver/parquet/patients.parquet").cache()
admissions_s    = spark.read.parquet("s3a://silver/parquet/admissions.parquet").cache()
icustays_s      = spark.read.parquet("s3a://silver/parquet/icustays.parquet").cache()
labevents_s     = spark.read.parquet("s3a://silver/parquet/labevents.parquet").persist(StorageLevel.MEMORY_AND_DISK)
prescriptions_s = spark.read.parquet("s3a://silver/parquet/prescriptions.parquet").cache()
transfers_s     = spark.read.parquet("s3a://silver/parquet/transfers.parquet").cache()
services_s      = spark.read.parquet("s3a://silver/parquet/services.parquet").cache()

# ── Jointure principale ───────────────────────────────────────────
df_gold = admissions_s.join(
    patients_s.select("SUBJECT_ID", "GENDER", "DOB", "EXPIRE_FLAG"),
    on="SUBJECT_ID", how="left"
).join(
    icustays_s.select("HADM_ID", "ICUSTAY_ID", "FIRST_CAREUNIT", "LOS"),
    on="HADM_ID", how="left"
)

# ── Calculer LOSdays ──────────────────────────────────────────────
df_gold = df_gold.withColumn(
    "LOSdays",
    F.round(
        (F.unix_timestamp("DISCHTIME") - F.unix_timestamp("ADMITTIME")) / 86400, 2
    )
)

# ── Agréger LABEVENTS par séjour ──────────────────────────────────
labs_agg = labevents_s.groupBy("HADM_ID").agg(
    F.count("ROW_ID").alias("num_labs"),
    F.countDistinct("ITEMID").alias("num_distinct_labs"),
    F.sum(F.when(F.col("FLAG") == "abnormal", 1).otherwise(0)).alias("num_abnormal_labs")
)

# ── Agréger PRESCRIPTIONS par séjour ─────────────────────────────
rx_agg = prescriptions_s.groupBy("HADM_ID").agg(
    F.count("ROW_ID").alias("num_rx"),
    F.countDistinct("DRUG").alias("num_distinct_drugs")
)

# ── Joindre les agrégations ───────────────────────────────────────
df_gold = df_gold \
    .join(labs_agg, on="HADM_ID", how="left") \
    .join(rx_agg,   on="HADM_ID", how="left") \
    .fillna(0, subset=["num_labs", "num_distinct_labs",
                       "num_abnormal_labs", "num_rx",
                       "num_distinct_drugs", "LOS"])

# ── Features métier ───────────────────────────────────────────────
# Feature 1 : score de complexité
df_gold = df_gold.withColumn(
    "complexity_score",
    F.round((F.col("num_labs") + F.col("num_rx") + F.col("num_distinct_drugs")) / 3, 2)
)

# Feature 2 : taux de labs anormaux
df_gold = df_gold.withColumn(
    "abnormal_lab_rate",
    F.round(
        F.when(F.col("num_labs") > 0,
            F.col("num_abnormal_labs") / F.col("num_labs")
        ).otherwise(0), 3
    )
)

# Feature 3 : patient à risque
df_gold = df_gold.withColumn(
    "high_risk",
    F.when(F.col("LOSdays") > 6.5, 1).otherwise(0)
)

# Feature 4 : durée moyenne par type d'admission
window_admit = Window.partitionBy("ADMISSION_TYPE")
df_gold = df_gold.withColumn(
    "avg_los_by_admit_type",
    F.round(F.avg("LOSdays").over(window_admit), 2)
)

# Feature 5 : durée moyenne par service ICU
window_icu = Window.partitionBy("FIRST_CAREUNIT")
df_gold = df_gold.withColumn(
    "avg_los_by_careunit",
    F.round(F.avg("LOSdays").over(window_icu), 2)
)

# ── Sauvegarder Gold dans MinIO ───────────────────────────────────
df_gold.write.mode("overwrite").parquet("s3a://gold/parquet/mimic_features.parquet")

print(f"✅ Gold sauvegardé dans MinIO")
print(f"   Lignes   : {df_gold.count():,}")
print(f"   Colonnes : {len(df_gold.columns)}")

✅ Gold sauvegardé dans MinIO
   Lignes   : 122,756
   Colonnes : 36
